In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from ib_async import IB

In [2]:
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "Market Data" / "Cleaned"
DATA_DIR.mkdir(parents=True, exist_ok=True)

TARGETS_PATH = DATA_DIR / "etf_targets_monthly.parquet"

RECON_CSV = DATA_DIR / "paper_reconciliation.csv"
RECON_PARQUET = DATA_DIR / "paper_reconciliation.parquet"
BLOTTER_CSV = DATA_DIR / "paper_order_blotter.csv"
BLOTTER_PARQUET = DATA_DIR / "paper_order_blotter.parquet"
ORDERS_CSV = DATA_DIR / "paper_orders_only.csv"
ORDERS_PARQUET = DATA_DIR / "paper_orders_only.parquet"

In [3]:
try:
    if ib.isConnected():
        ib.disconnect()
except NameError:
    pass

ib = IB()
await ib.connectAsync("127.0.0.1", 4004, clientId=2)

print("Connected:", ib.isConnected())
print("Managed accounts:", ib.managedAccounts())

Connected: True
Managed accounts: ['DUP317554']


In [4]:
targets = pd.read_parquet(TARGETS_PATH)
targets["date"] = pd.to_datetime(targets["date"])
targets = targets.sort_values(["date", "symbol"]).reset_index(drop=True)

latest_date = targets["date"].max()
latest_targets = targets[targets["date"] == latest_date].copy()

display(
    latest_targets[
        ["date", "symbol", "bucket", "close", "target_weight"]
    ].sort_values("target_weight", ascending=False)
)

print("Latest rebalance date:", latest_date)

,date,symbol,bucket,close,target_weight
4629,2026-03-06,QQQ,core_us,599.75,0.300000
4632,2026-03-06,SPY,core_us,672.38,0.300000
4641,2026-03-06,VPU,us_sector,201.60,0.050000
4639,2026-03-06,VIS,us_sector,326.26,0.050000
4638,2026-03-06,VHT,us_sector,282.15,0.050000
4637,2026-03-06,VFH,us_sector,123.43,0.050000
4642,2026-03-06,VWO,international,54.47,0.033333
4636,2026-03-06,VEA,international,65.28,0.033333
4628,2026-03-06,INDA,international,49.99,0.033333
4625,2026-03-06,BIL,defensive,91.44,0.012500


Latest rebalance date: 2026-03-06 00:00:00


In [5]:
positions = ib.positions()

if len(positions) == 0:
    positions_df = pd.DataFrame(columns=["symbol", "current_shares", "avg_cost"])
else:
    positions_df = pd.DataFrame(
        [
            {
                "symbol": getattr(p.contract, "symbol", None),
                "current_shares": float(p.position),
                "avg_cost": float(p.avgCost),
            }
            for p in positions
        ]
    )

    positions_df = (
        positions_df.groupby("symbol", as_index=False)
        .agg(
            current_shares=("current_shares", "sum"),
            avg_cost=("avg_cost", "mean"),
        )
    )

display(positions_df)

,symbol,current_shares,avg_cost


In [6]:
acct = await ib.accountSummaryAsync()

netliq_rows = [row for row in acct if getattr(row, "tag", None) == "NetLiquidation"]
cash_rows = [row for row in acct if getattr(row, "tag", None) == "TotalCashValue"]

if len(netliq_rows) == 0:
    raise ValueError("Could not find NetLiquidation in IBKR account summary.")

portfolio_value = float(netliq_rows[0].value)
cash_value = float(cash_rows[0].value) if len(cash_rows) > 0 else np.nan

print("Portfolio value:", portfolio_value)
print("Cash value:", cash_value)

Portfolio value: 1000086.35
Cash value: 1000000.0


In [7]:
recon = latest_targets.merge(
    positions_df[["symbol", "current_shares"]] if len(positions_df) > 0 else pd.DataFrame(columns=["symbol", "current_shares"]),
    on="symbol",
    how="left"
)

recon["current_shares"] = recon["current_shares"].fillna(0).astype(int)

/var/folders/wk/189ck3455t5d1pp0p1ydbpxw0000gp/T/ipykernel_52510/2652013018.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  recon["current_shares"] = recon["current_shares"].fillna(0).astype(int)


In [8]:
recon["target_dollars"] = portfolio_value * recon["target_weight"]

recon["target_shares"] = np.floor(
    recon["target_dollars"] / recon["close"]
).astype(int)

recon["delta_shares"] = recon["target_shares"] - recon["current_shares"]

recon["side"] = np.where(
    recon["delta_shares"] > 0,
    "BUY",
    np.where(recon["delta_shares"] < 0, "SELL", "HOLD")
)

recon["abs_delta_shares"] = recon["delta_shares"].abs()
recon["estimated_trade_dollars"] = recon["abs_delta_shares"] * recon["close"]

In [9]:
recon = recon[
    [
        "date",
        "symbol",
        "bucket",
        "close",
        "target_weight",
        "target_dollars",
        "target_shares",
        "current_shares",
        "delta_shares",
        "side",
        "estimated_trade_dollars",
        "abs_delta_shares"
    ]
].copy()

recon = recon.sort_values(
    ["side", "estimated_trade_dollars"],
    ascending=[True, False]
).reset_index(drop=True)

display(recon)

,date,symbol,bucket,close,target_weight,target_dollars,target_shares,current_shares,delta_shares,side,estimated_trade_dollars,abs_delta_shares
0,2026-03-06,SPY,core_us,672.38,0.300000,300025.905000,446,0,446,BUY,299881.48,446
1,2026-03-06,QQQ,core_us,599.75,0.300000,300025.905000,500,0,500,BUY,299875.00,500
2,2026-03-06,VPU,us_sector,201.60,0.050000,50004.317500,248,0,248,BUY,49996.80,248
3,2026-03-06,VFH,us_sector,123.43,0.050000,50004.317500,405,0,405,BUY,49989.15,405
4,2026-03-06,VHT,us_sector,282.15,0.050000,50004.317500,177,0,177,BUY,49940.55,177
5,2026-03-06,VIS,us_sector,326.26,0.050000,50004.317500,153,0,153,BUY,49917.78,153
6,2026-03-06,VWO,international,54.47,0.033333,33336.211667,612,0,612,BUY,33335.64,612
7,2026-03-06,INDA,international,49.99,0.033333,33336.211667,666,0,666,BUY,33293.34,666
8,2026-03-06,VEA,international,65.28,0.033333,33336.211667,510,0,510,BUY,33292.80,510
9,2026-03-06,SHY,defensive,82.73,0.012500,12501.079375,151,0,151,BUY,12492.23,151


In [10]:
MIN_TRADE_DOLLARS = 1000.0

orders_only = recon[
    (recon["side"] != "HOLD") &
    (recon["abs_delta_shares"] > 0) &
    (recon["estimated_trade_dollars"] >= MIN_TRADE_DOLLARS)
].copy().reset_index(drop=True)

display(orders_only)

,date,symbol,bucket,close,target_weight,target_dollars,target_shares,current_shares,delta_shares,side,estimated_trade_dollars,abs_delta_shares
0,2026-03-06,SPY,core_us,672.38,0.300000,300025.905000,446,0,446,BUY,299881.48,446
1,2026-03-06,QQQ,core_us,599.75,0.300000,300025.905000,500,0,500,BUY,299875.00,500
2,2026-03-06,VPU,us_sector,201.60,0.050000,50004.317500,248,0,248,BUY,49996.80,248
3,2026-03-06,VFH,us_sector,123.43,0.050000,50004.317500,405,0,405,BUY,49989.15,405
4,2026-03-06,VHT,us_sector,282.15,0.050000,50004.317500,177,0,177,BUY,49940.55,177
5,2026-03-06,VIS,us_sector,326.26,0.050000,50004.317500,153,0,153,BUY,49917.78,153
6,2026-03-06,VWO,international,54.47,0.033333,33336.211667,612,0,612,BUY,33335.64,612
7,2026-03-06,INDA,international,49.99,0.033333,33336.211667,666,0,666,BUY,33293.34,666
8,2026-03-06,VEA,international,65.28,0.033333,33336.211667,510,0,510,BUY,33292.80,510
9,2026-03-06,SHY,defensive,82.73,0.012500,12501.079375,151,0,151,BUY,12492.23,151


In [11]:
print("Weight sum:", recon["target_weight"].sum())
print("Target dollars total:", recon["target_dollars"].sum())
print("Order count:", len(orders_only))
print("Buy dollars:", orders_only.loc[orders_only["side"] == "BUY", "estimated_trade_dollars"].sum())
print("Sell dollars:", orders_only.loc[orders_only["side"] == "SELL", "estimated_trade_dollars"].sum())

Weight sum: 1.0
Target dollars total: 1000086.35
Order count: 18
Buy dollars: 999044.33
Sell dollars: 0.0


In [12]:
print("Weight sum:", recon["target_weight"].sum())
print("Target dollars total:", recon["target_dollars"].sum())
print("Total target shares:", recon["target_shares"].sum())
print("Order count:", len(orders_only))
print("Buy dollars:", orders_only.loc[orders_only["side"] == "BUY", "estimated_trade_dollars"].sum())
print("Sell dollars:", orders_only.loc[orders_only["side"] == "SELL", "estimated_trade_dollars"].sum())

Weight sum: 1.0
Target dollars total: 1000086.35
Total target shares: 5382
Order count: 18
Buy dollars: 999044.33
Sell dollars: 0.0


In [13]:
recon.to_csv(RECON_CSV, index=False)
recon.to_parquet(RECON_PARQUET, index=False)

recon.to_csv(BLOTTER_CSV, index=False)
recon.to_parquet(BLOTTER_PARQUET, index=False)

orders_only.to_csv(ORDERS_CSV, index=False)
orders_only.to_parquet(ORDERS_PARQUET, index=False)

print("Saved:")
print(RECON_CSV)
print(RECON_PARQUET)
print(BLOTTER_CSV)
print(BLOTTER_PARQUET)
print(ORDERS_CSV)
print(ORDERS_PARQUET)

Saved:
/Users/tradingworkspace/TradingWorkspace/data-engineering/Market Data/Cleaned/paper_reconciliation.csv
/Users/tradingworkspace/TradingWorkspace/data-engineering/Market Data/Cleaned/paper_reconciliation.parquet
/Users/tradingworkspace/TradingWorkspace/data-engineering/Market Data/Cleaned/paper_order_blotter.csv
/Users/tradingworkspace/TradingWorkspace/data-engineering/Market Data/Cleaned/paper_order_blotter.parquet
/Users/tradingworkspace/TradingWorkspace/data-engineering/Market Data/Cleaned/paper_orders_only.csv
/Users/tradingworkspace/TradingWorkspace/data-engineering/Market Data/Cleaned/paper_orders_only.parquet


In [14]:
ib.disconnect()
print("Disconnected.")

Disconnected.


In [15]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "Market Data" / "Cleaned"
DATA_DIR.mkdir(parents=True, exist_ok=True)

orders_only_parquet = DATA_DIR / "orders_only.parquet"
orders_only_csv = DATA_DIR / "orders_only.csv"

orders_only.to_parquet(orders_only_parquet, index=False)
orders_only.to_csv(orders_only_csv, index=False)

print(orders_only_parquet)
print(orders_only_csv)

/Users/tradingworkspace/TradingWorkspace/data-engineering/Market Data/Cleaned/orders_only.parquet
/Users/tradingworkspace/TradingWorkspace/data-engineering/Market Data/Cleaned/orders_only.csv
